# ✅ Action Detection — High Accuracy Version
### Fixes: Better model, data augmentation, proper training pipeline

In [ ]:
# Cell 1 — Imports
import cv2
import numpy as np
import os
import random
import collections
import threading
import subprocess
from matplotlib import pyplot as plt
import time
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.utils import to_categorical
from collections import Counter
print('✅ All imports done!')

In [ ]:
# Cell 2 — MediaPipe Setup
mp_holistic   = mp.solutions.holistic
mp_drawing    = mp.solutions.drawing_utils
mp_face_mesh  = mp.solutions.face_mesh
print('✅ MediaPipe ready!')

In [ ]:
# Cell 3 — Core Functions
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() \
           if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() \
           if results.face_landmarks else np.zeros(468*3)
    lh   = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() \
           if results.left_hand_landmarks else np.zeros(21*3)
    rh   = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() \
           if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

def get_recorded_actions(data_path):
    if not os.path.exists(data_path):
        return np.array([])
    return np.array(sorted([d for d in os.listdir(data_path)
                            if os.path.isdir(os.path.join(data_path, d))]))

print('✅ Functions ready!')

In [ ]:
# Cell 4 — Config
DATA_PATH       = 'MP_Data'
sequence_length = 30
actions         = get_recorded_actions(DATA_PATH)
print(f'✅ Found {len(actions)} signs: {list(actions)}')

# Check sequences per action
for action in actions:
    path  = os.path.join(DATA_PATH, action)
    count = len(os.listdir(path))
    print(f'   {action}: {count} sequences')

In [ ]:
# Cell 5 — ✅ IMPROVED DATA LOADING with augmentation
# KEY FIX: Data augmentation creates more training variety = higher accuracy!

def augment_sequence(sequence, noise_factor=0.002):
    """Add tiny random noise to create augmented samples"""
    return sequence + np.random.normal(0, noise_factor, sequence.shape)

np.random.seed(42)
random.seed(42)

label_map = {label: num for num, label in enumerate(actions)}
print(f'Label map: {label_map}')

sequences, labels = [], []

for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    available   = sorted([int(s) for s in os.listdir(action_path)
                          if os.path.isdir(os.path.join(action_path, s))])
    for sequence in available:
        window   = []
        seq_path = os.path.join(DATA_PATH, action, str(sequence))
        all_exist = all(os.path.exists(os.path.join(seq_path, f'{f}.npy'))
                        for f in range(sequence_length))
        if all_exist:
            for frame_num in range(sequence_length):
                res = np.load(os.path.join(seq_path, f'{frame_num}.npy'))
                window.append(res)
            seq_arr = np.array(window)
            sequences.append(seq_arr)
            labels.append(label_map[action])

            # ✅ AUGMENTATION: add 2 noisy copies per sequence
            for _ in range(2):
                sequences.append(augment_sequence(seq_arr))
                labels.append(label_map[action])

X            = np.array(sequences)
labels_array = np.array(labels)

print(f'\n✅ Class counts after augmentation: {Counter(labels_array)}')
print(f'✅ Total samples: {len(X)}')
print(f'✅ Shape: {X.shape}')

y = to_categorical(labels_array).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=labels_array
)
print(f'✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
# Cell 6 — ✅ IMPROVED MODEL ARCHITECTURE
# KEY FIXES:
# 1. Added BatchNormalization — stabilizes training
# 2. Better Dropout values — less overfitting
# 3. Added L2 regularization — prevents memorization
# 4. Larger LSTM units — more capacity to learn

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
)
import tensorflow as tf

tf.random.set_seed(42)

model = Sequential([
    Input(shape=(30, 1662)),

    # ✅ LSTM Block 1
    LSTM(128, return_sequences=True, activation='tanh',
         kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),

    # ✅ LSTM Block 2
    LSTM(256, return_sequences=True, activation='tanh',
         kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.3),

    # ✅ LSTM Block 3
    LSTM(128, return_sequences=False, activation='tanh',
         kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),

    # ✅ Dense layers
    Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),
    Dense(len(actions), activation='softmax')
])

# ✅ Better optimizer with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

model.summary()
print(f'\n✅ Model built for {len(actions)} signs: {list(actions)}')

In [ ]:
# Cell 7 — ✅ IMPROVED TRAINING
# KEY FIXES:
# 1. EarlyStopping with patience=30 — stops at best point
# 2. ReduceLROnPlateau — slows learning rate when stuck
# 3. Saves BEST weights automatically
# 4. More epochs = more chances to learn

log_dir    = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

early_stop = EarlyStopping(
    monitor='val_categorical_accuracy',
    patience=30,
    restore_best_weights=True,
    mode='max',
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_action.h5',
    monitor='val_categorical_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_categorical_accuracy',
    patience=15,
    factor=0.5,
    min_lr=1e-7,
    mode='max',
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=500,
    batch_size=16,
    shuffle=True,
    validation_split=0.15,
    callbacks=[tb_callback, early_stop, checkpoint, reduce_lr],
    verbose=1
)

model.save('action.h5')
print('\n✅ Training complete! Model saved!')

In [ ]:
# Cell 8 — Plot Training History
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['categorical_accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_categorical_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()
print('✅ Plot saved!')

In [ ]:
# Cell 9 — Load Best Weights + Evaluate
from tensorflow.keras.models import load_model

# Load best saved model
model = load_model('best_action.h5')
print('✅ Best model loaded!')

# Evaluate
yhat_probs = model.predict(X_test, verbose=0)
ytrue      = np.argmax(y_test, axis=1)
yhat       = np.argmax(yhat_probs, axis=1)

acc = accuracy_score(ytrue, yhat)
print(f'\n✅ Test Accuracy: {acc*100:.2f}%')
print('\n✅ Classification Report:')
print(classification_report(ytrue, yhat, target_names=actions))

In [ ]:
# Cell 10 — Setup for Real-Time Detection
random.seed(42)
colors = [
    (random.randint(50,230), random.randint(50,230), random.randint(50,230))
    for _ in actions
]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0, 60+num*45), (200, 90+num*45), (50,50,50), -1)
        cv2.rectangle(output_frame, (0, 60+num*45), (int(prob*200), 90+num*45), colors[num], -1)
        cv2.putText(output_frame, f'{actions[num]} {prob*100:.0f}%',
                    (5, 83+num*45), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)
    return output_frame

def speak(word):
    def run():
        subprocess.Popen(
            ['powershell', '-Command',
             f'Add-Type -AssemblyName System.Speech;'
             f'$s=New-Object System.Speech.Synthesis.SpeechSynthesizer;'
             f'$s.Rate=5;$s.Speak("{word}")'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    threading.Thread(target=run, daemon=True).start()

def clean_sentence(sentence):
    if not sentence: return sentence
    cleaned = [sentence[0]]
    for word in sentence[1:]:
        if word != cleaned[-1]: cleaned.append(word)
    return cleaned

print('✅ Real-time detection ready!')
print(f'✅ Actions: {list(actions)}')

In [ ]:
# Cell 11 — ✅ IMPROVED Real-Time Detection
# KEY FIXES:
# 1. avg_gap check — ensures confident prediction not random
# 2. Smoothed predictions using deque
# 3. Per-action thresholds tuned
# 4. Cooldown prevents false repeats

# ✅ Tune these thresholds based on your accuracy report
THRESHOLDS = {
    'good':   0.85,
    'hello':  0.85,
    'thanks': 0.85,
}

MIN_AGREE             = 15   # frames that must agree
COOLDOWN              = 30   # frames to wait after detection
GAP_THRESHOLD         = 0.20 # min gap between top-2 probabilities

cooldown_counter      = 0
sequence              = []
sentence              = []
predictions           = collections.deque(maxlen=MIN_AGREE)
probabilities         = collections.deque(maxlen=MIN_AGREE)
last_confirmed_action = ''
last_spoken           = ''
frame_count           = 0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

cv2.namedWindow('OpenCV Feed', cv2.WINDOW_NORMAL)
cv2.moveWindow('OpenCV Feed', 0, 0)

with mp_holistic.Holistic(
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6,
        model_complexity=1) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        if frame_count % 2 != 0:
            continue

        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        hand_visible = (results.left_hand_landmarks is not None or
                        results.right_hand_landmarks is not None)

        if not hand_visible:
            sequence = []
            predictions.clear()
            probabilities.clear()
            cv2.rectangle(image, (0,0), (320,40), (0,0,0), -1)
            cv2.putText(image, 'Show hand...', (5,28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100,100,100), 2)
        else:
            keypoints = extract_keypoints(results)
            sequence.append(keypoints)
            sequence = sequence[-30:]

            if cooldown_counter > 0:
                cooldown_counter -= 1
            elif len(sequence) == 30:
                res = model.predict(
                    np.expand_dims(sequence, axis=0), verbose=0)[0]

                predicted_idx    = np.argmax(res)
                predicted_action = actions[predicted_idx]
                confidence       = float(res[predicted_idx])
                threshold        = THRESHOLDS.get(predicted_action, 0.85)

                # ✅ Gap between top 2 predictions
                sorted_probs = sorted(res, reverse=True)
                gap = sorted_probs[0] - sorted_probs[1]

                predictions.append(predicted_idx)
                probabilities.append(res)
                image = prob_viz(res, actions, image, colors)

                if len(predictions) == MIN_AGREE:
                    all_same = len(set(predictions)) == 1

                    if not all_same:
                        predictions.clear()
                        probabilities.clear()
                    else:
                        avg_conf = float(np.mean(
                            [p[predicted_idx] for p in probabilities]))
                        avg_gap  = float(np.mean([
                            sorted(p, reverse=True)[0] - sorted(p, reverse=True)[1]
                            for p in probabilities
                        ]))

                        if (avg_conf > threshold and
                            confidence > threshold and
                            avg_gap > GAP_THRESHOLD and
                            predicted_action != last_confirmed_action):

                            sentence.append(predicted_action)
                            speak(predicted_action)
                            last_spoken           = predicted_action
                            last_confirmed_action = predicted_action
                            print(f'✅ {predicted_action} ({confidence*100:.1f}%) gap={gap:.2f}')
                            sequence      = []
                            predictions.clear()
                            probabilities.clear()
                            cooldown_counter = COOLDOWN
                        else:
                            predictions.clear()
                            probabilities.clear()

            sentence = clean_sentence(sentence)
            if len(sentence) > 3:
                sentence = sentence[-3:]

            cv2.rectangle(image, (0,0), (320,40), (0,0,0), -1)
            cv2.putText(image,
                        ' '.join(sentence) if sentence else 'Waiting...',
                        (5,28), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (255,255,255), 2)

        buf_len = len(sequence)
        cv2.rectangle(image, (0,220), (int(buf_len/30*320), 240), (0,255,100), -1)
        cv2.putText(image, f'Buf:{buf_len}/30  Said:{last_spoken}',
                    (5,235), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,0,0), 1)

        cv2.imshow('OpenCV Feed', image)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
        elif key == ord('c'):
            sentence              = []
            sequence              = []
            last_confirmed_action = ''
            last_spoken           = ''
            cooldown_counter      = 0
            predictions.clear()
            probabilities.clear()

cap.release()
cv2.destroyAllWindows()
print('✅ Detection stopped!')